# YOLOv8n on COCO128 — Kubeflow Notebook

This notebook runs **inside a Kubeflow Notebook server** — a Kubernetes-managed Jupyter pod.

| Step | What you learn |
|---|---|
| 1 | Install deps inside the notebook pod |
| 2 | Kubeflow Notebook environment info |
| 3 | Download & explore COCO128 |
| 4 | PyTorch inference |
| 5 | Export to ONNX |
| 6 | PyTorch vs ONNX benchmark |
| 7 | Evaluate mAP on val set |
| 8 | Per-class mAP breakdown |

> **Kubeflow Notebook fact:** `kubectl -n yolo-project get notebook yolo-notebook`
> shows this server as a Kubernetes CR. The PVC `yolo-workspace` backs
> `/home/jovyan` so files survive pod restarts.

In [ ]:
# Cell 1 — Install dependencies
import sys
!{sys.executable} -m pip install -q ultralytics==8.2.0 onnx==1.16.0 onnxruntime==1.18.0
print('✅ Dependencies installed')

In [ ]:
# Cell 2 — Show Kubeflow Notebook environment info
import os, socket
from pathlib import Path

print('=== Kubeflow Notebook Pod Info ===')
print(f'Pod name   : {socket.gethostname()}')
print(f'Namespace  : {os.environ.get("POD_NAMESPACE", "yolo-project")}')
print(f'Home dir   : {Path.home()}')
print(f'PVC mount  : /home/jovyan  ← backed by yolo-workspace PVC')
print()
print('Check this pod from your terminal:')
print('  kubectl -n yolo-project get notebook yolo-notebook')
print('  kubectl -n yolo-project get pod -l notebook-name=yolo-notebook')
!df -h /home/jovyan 2>/dev/null || echo 'df not available'

In [ ]:
# Cell 3 — Download COCO128
from pathlib import Path
from ultralytics import settings
from ultralytics.data.utils import check_det_dataset

DATASETS_DIR = Path('/home/jovyan/datasets')
DATASETS_DIR.mkdir(exist_ok=True)
settings.update({'datasets_dir': str(DATASETS_DIR)})

data_dict = check_det_dataset('coco128.yaml')
dataset_path = Path(data_dict['path'])
names = data_dict.get('names', {})

print(f'Dataset  : {dataset_path}')
print(f'Classes  : {data_dict.get("nc")} total')
print(f'Sample   : {list(names.values())[:8]}...')
print(f'Train    : {data_dict.get("train")}')
print(f'Val      : {data_dict.get("val")}')

In [ ]:
# Cell 4 — Plot class distribution
import glob, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

label_dir = dataset_path / 'labels' / 'train'
class_counts = {}
for lf in glob.glob(str(label_dir / '*.txt')):
    with open(lf) as f:
        for line in f:
            cls = int(line.split()[0])
            class_counts[cls] = class_counts.get(cls, 0) + 1

top20 = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)[:20]
labels = [names.get(c, str(c)) for c, _ in top20]
values = [v for _, v in top20]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(labels[::-1], values[::-1], color='steelblue')
ax.set_xlabel('Annotation count')
ax.set_title('Top 20 Classes in COCO128')
plt.tight_layout()
out = '/home/jovyan/class_dist.png'
plt.savefig(out, dpi=100); plt.close()
display(Image.open(out))

In [ ]:
# Cell 5 — Load YOLOv8n and run PyTorch inference
import glob, random, time
from ultralytics import YOLO

image_files = glob.glob(str(dataset_path / 'images' / 'train' / '*.jpg'))
print(f'Total images: {len(image_files)}')

model_pt = YOLO('yolov8n.pt')   # auto-downloads ~6 MB
print(f'Parameters : {sum(p.numel() for p in model_pt.model.parameters()):,}')

test_imgs = random.sample(image_files, 3)
t0 = time.time()
results = model_pt.predict(test_imgs, conf=0.25, verbose=False)
pt_ms = (time.time() - t0) * 1000

for r, p in zip(results, test_imgs):
    print(f'  {Path(p).name}: {len(r.boxes)} detections')
print(f'PyTorch: {pt_ms:.0f}ms total  ({pt_ms/3:.0f}ms/img)')

In [ ]:
# Cell 6 — Export to ONNX
import onnx
import onnxruntime as ort
import numpy as np

export_path = model_pt.export(
    format='onnx', imgsz=640, opset=12, simplify=True, dynamic=False
)
onnx.checker.check_model(onnx.load(export_path))
print(f'ONNX saved  : {export_path}')
print(f'File size   : {Path(export_path).stat().st_size/1e6:.1f} MB')

sess = ort.InferenceSession(export_path, providers=['CPUExecutionProvider'])
dummy = np.zeros((1, 3, 640, 640), dtype=np.float32)
out = sess.run(None, {'images': dummy})
print(f'Output shape: {out[0].shape}')
print('  → 84 = 4 box coords + 80 COCO class scores')
print('  → 8400 = candidate detections per image')

In [ ]:
# Cell 7 — Benchmark PyTorch vs ONNX
from PIL import Image as PILImage

def preprocess(img_path, size=640):
    img = PILImage.open(img_path).convert('RGB').resize((size, size))
    arr = np.array(img, dtype=np.float32) / 255.0
    return arr.transpose(2, 0, 1)[np.newaxis, ...]

bench = test_imgs[0]
inp   = preprocess(bench)
N     = 10

_ = sess.run(None, {'images': inp})   # warm up

t0 = time.time()
for _ in range(N): sess.run(None, {'images': inp})
onnx_ms = (time.time() - t0) * 1000 / N

t0 = time.time()
for _ in range(N): model_pt.predict(bench, conf=0.25, verbose=False)
pt_ms2 = (time.time() - t0) * 1000 / N

print(f'=== Benchmark ({N} runs each) ===')
print(f'PyTorch      : {pt_ms2:7.1f} ms/img')
print(f'ONNX Runtime : {onnx_ms:7.1f} ms/img')
print(f'Speedup      : {pt_ms2/onnx_ms:.2f}x')

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['PyTorch', 'ONNX'], [pt_ms2, onnx_ms], color=['#e07b39', '#4c72b0'])
ax.set_ylabel('ms / image')
ax.set_title('Inference Speed: PyTorch vs ONNX')
for i, v in enumerate([pt_ms2, onnx_ms]):
    ax.text(i, v + 1, f'{v:.0f}ms', ha='center')
plt.tight_layout()
plt.savefig('/home/jovyan/benchmark.png', dpi=100); plt.close()
display(PILImage.open('/home/jovyan/benchmark.png'))

In [ ]:
# Cell 8 — Evaluate mAP on COCO128 val split
print('Running evaluation... (~1 min with 32 GB RAM / 8 CPUs)')
val_results = model_pt.val(
    data='coco128.yaml',
    imgsz=640,
    batch=16,       # larger batch since we have more RAM
    conf=0.25,
    iou=0.6,
    device='cpu',
    verbose=False,
    plots=False,
)

print(f'\n=== Evaluation Results (COCO128 val) ===')
print(f'mAP50     : {val_results.box.map50:.4f}')
print(f'mAP50-95  : {val_results.box.map:.4f}')
print(f'Precision : {val_results.box.mp:.4f}')
print(f'Recall    : {val_results.box.mr:.4f}')
print()
print('These same metrics are logged by the KFP evaluate_model component')
print('and appear in KFP UI → Runs → your run → Artifacts tab.')

In [ ]:
# Cell 9 — Per-class mAP chart
try:
    per_class_idx  = val_results.box.ap_class_index
    per_class_map50 = val_results.box.ap50
    if per_class_map50 is not None and len(per_class_map50) > 0:
        pairs = sorted(
            zip(per_class_idx, per_class_map50),
            key=lambda x: x[1], reverse=True
        )[:20]
        cls_labels = [names.get(c, str(c)) for c, _ in pairs]
        cls_vals   = [v for _, v in pairs]

        fig, ax = plt.subplots(figsize=(9, 5))
        ax.barh(cls_labels[::-1], cls_vals[::-1], color='steelblue')
        ax.axvline(x=val_results.box.map50, color='red', linestyle='--',
                   label=f'mean mAP50 = {val_results.box.map50:.3f}')
        ax.set_xlabel('mAP50')
        ax.set_title('Per-Class mAP50 (Top 20) — YOLOv8n on COCO128')
        ax.legend()
        plt.tight_layout()
        plt.savefig('/home/jovyan/per_class_map.png', dpi=100); plt.close()
        display(PILImage.open('/home/jovyan/per_class_map.png'))
    else:
        print('Per-class data not available.')
except Exception as e:
    print(f'Could not plot per-class mAP: {e}')

In [ ]:
# Cell 10 — Summary
print('=' * 55)
print(' Kubeflow Notebook Exploration — Complete!')
print('=' * 55)
print()
print('What ran in this notebook:')
print('  ✅ Downloaded COCO128 (128 images, 80 classes)')
print('  ✅ Explored class distribution')
print('  ✅ Loaded YOLOv8n.pt and ran PyTorch inference')
print('  ✅ Exported to ONNX (opset 12, 640×640)')
print('  ✅ Benchmarked PyTorch vs ONNX speed')
print('  ✅ Evaluated mAP50/mAP50-95 on val split')
print('  ✅ Plotted per-class mAP breakdown')
print()
print('Kubeflow feature used:')
print('  → This IS a Kubernetes pod (Notebook Controller CR)')
print('  → Check: kubectl -n yolo-project get notebook yolo-notebook')
print('  → PVC keeps /home/jovyan alive across restarts')
print()
print('Next: run the KFP pipeline and watch the same steps')
print('run as isolated containers in the KFP UI!')
print('  bash scripts/03-compile-pipeline.sh')